In [45]:
# TEXT GENERATOR USING SIMPLE RNN

# Import NumPy library → used for handling arrays and numerical operations
import numpy as np

# Import TensorFlow → main library for building deep learning models
import tensorflow as tf

# Import Sequential → used to build model layer-by-layer (simple neural network structure)
from tensorflow.keras.models import Sequential

# Import layers:
# SimpleRNN → learns sequence patterns (like sentences)
# Dense → fully connected layer (used for output)
# Embedding → converts words into numerical vectors
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding

# Import Tokenizer → converts text (words) into numbers
from tensorflow.keras.preprocessing.text import Tokenizer

# Import pad_sequences → makes all sequences same length by adding padding
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Import pickle → used to save and load objects (like tokenizer) for future use
import pickle

In [46]:
# Load dataset

# Open the file "dataset.txt" in read mode ("r")
# encoding="utf-8" ensures proper reading of all characters (including special symbols)
with open("dataset.txt", "r", encoding="utf-8") as f:
    
    # Read all text from the file
    # .lower() converts all text to lowercase (helps in consistent processing)
    data = f.read().lower()

In [47]:
# Tokenizer

# Create a tokenizer object → used to convert words into numbers
tokenizer = Tokenizer()

# Fit tokenizer on the dataset
# It learns all unique words and assigns each word a number (index)
tokenizer.fit_on_texts([data])

# Get total number of unique words in dataset
# +1 is added because indexing starts from 1 (0 is reserved for padding)
total_words = len(tokenizer.word_index) + 1

In [48]:
# Create sequences

# Empty list to store input sequences
input_sequences = []

# Loop through each line in the dataset (split by new line)
for line in data.split("\n"):
    
    # Convert line (text) into list of numbers using tokenizer
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    # Create sequences from each line
    for i in range(1, len(token_list)):
    
        # Take words from start up to i+1 and add to list
        # This helps model learn patterns step-by-step
        input_sequences.append(token_list[:i+1]); 

In [49]:
# Padding

# Find the maximum length among all sequences
# This helps us decide the uniform length for all inputs
max_seq_len = max(len(seq) for seq in input_sequences)

# Apply padding to all sequences
# maxlen=max_seq_len → make all sequences same length
# padding='pre' → add zeros at the beginning of shorter sequences
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

In [50]:
# Split X and y

# X (input features):
# Take all elements of each sequence EXCEPT the last one
# This will be used as input to the model
X = input_sequences[:, :-1]

# y (target/output):
# Take only the LAST element of each sequence
# This is the word the model needs to predict
y = input_sequences[:, -1]

In [51]:
# One-hot encoding

# Convert y (target word indices) into one-hot encoded format
# Example: if total_words = 5 and y = 2 → [0, 0, 1, 0, 0]
# This helps the model treat output as a classification problem
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [52]:
# RNN Model (NO LSTM)

# Create a Sequential model (layer-by-layer neural network)
model = Sequential([
    
    # Embedding layer:
    # total_words → size of vocabulary
    # 100 → each word converted into 100-dimensional vector
    # input_length → length of input sequence (X)
    Embedding(total_words, 100, input_length=max_seq_len-1),
    
    # First SimpleRNN layer:
    # 150 neurons → learns sequence patterns
    # return_sequences=True → passes full sequence to next RNN layer
    SimpleRNN(150, return_sequences=True),
    
    # Second SimpleRNN layer:
    # 100 neurons → further processes sequence
    # (no return_sequences → outputs only final result)
    SimpleRNN(100),
    
    # Dense layer (output layer):
    # total_words → predicts probability for each word
    # softmax → gives probability distribution
    Dense(total_words, activation='softmax')
])

# Compile the model:
# categorical_crossentropy → used for multi-class classification
# adam → optimizer (updates weights efficiently)
# accuracy → metric to check model performance
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [53]:
# Train

# Train the model using input (X) and target (y)
# epochs=100 → model will learn by going through data 100 times
# batch_size=32 → processes 32 samples at a time (faster & stable training)
# verbose=1 → shows training progress (loss, accuracy) in output
model.fit(X, y, epochs=100, batch_size=32, verbose=1)

Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.0522 - loss: 4.8831
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0763 - loss: 4.6355 
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0884 - loss: 4.4369 
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1165 - loss: 4.2563 
Epoch 5/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1526 - loss: 4.0630 
Epoch 6/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1928 - loss: 3.8929 
Epoch 7/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1847 - loss: 3.7541 
Epoch 8/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2490 - loss: 3.5466 
Epoch 9/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2851 - loss: 3.3650 
Epoch 10/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3173 - loss: 3.1978 
Epoch 11/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3414 - loss: 2.9986 
Epoch 12/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3815 - loss

In [54]:
# Save

# Save the trained model into a file
# "rnn_text_generator.h5" → file where model architecture + weights are stored
model.save("rnn_text_generator.h5")

# Save tokenizer object
# This is needed later to convert new text into numbers during prediction
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save maximum sequence length
# Required during text generation to maintain same input size
with open("max_seq_len.pkl", "wb") as f:
    pickle.dump(max_seq_len, f)

# Print message after training and saving is complete
print("RNN Training Complete!")

RNN Training Complete!
